# Hairpin Finder

This notebook detects and trims terminal DRC-like, hairpin-like, and palindrome-like structures from FASTA sequences, optionally normalizes sequence orientation against a reference genome, and reports terminal ITR candidates.

## How to use

1. Prepare and upload an input FASTA file.
2. Edit the configuration cell below, especially `INPUT_FASTA` and `OUTPUT_DIR`.
3. Optionally set `REFERENCE_FASTA` to orient final sequences against a reference genome.
4. Run all cells.
5. Results are written to `OUTPUT_DIR`.

## Current pipeline

1. **STEP 1: Smart DRC trimming**  
   Searches each terminal half of the sequence for a transition pattern: `opposite run -> overlap/none -> reversed run`.

2. **STEP 2: Hairpin / palindrome refinement**  
   Refines terminal trimming after DRC trimming.

3. **STEP 2.5: Reference-based orientation normalization**  
   Uses shared k-mers to decide whether the final sequence should be reverse-complemented.

4. **STEP 3: ITR detection**  
   Aligns the left terminal region against the reverse-complemented right terminal region and applies length, identity, adjusted-score, terminal-proximity, and informative-base filters.

5. **STEP 4: Output files**  
   Writes individual final FASTA files, a combined FASTA file, `summary_report.csv`, and an optional `drc_debug_report.csv`.


In [ ]:
!pip install -q biopython

import Bio

In [ ]:
# =========================
# User configuration
# =========================

INPUT_FASTA = "/path/Input_file.fasta"
OUTPUT_DIR = "/path/output_folder/"

# Optional reference genome FASTA for orientation normalization.
# Leave as None to skip reference-based orientation correction.
REFERENCE_FASTA = "/path/Reference_file.fasta"

# =========================
# Reference-based orientation settings
# =========================

ORIENTATION_K = 15
ORIENTATION_CHUNK_SIZE = 10000
ORIENTATION_N_CHUNKS = 25
ORIENTATION_STEP = 1
ORIENTATION_MIN_RATIO = 1.1 # 1.1 means a 10% score difference is required.

# =========================
# DRC trimming settings
# =========================

DRC_COARSE_STEP = 2000
DRC_FINE_STEP = 100
DRC_WINDOW_SIZE = 300
DRC_MIN_IDENTITY = 0.85
DRC_MIN_ALIGNMENT_COVERAGE = 0.80
DRC_MARGIN = 300

# A DRC window with too many ambiguous bases is ignored.
# This does not remove Ns from the sequence; it only avoids using low-confidence windows as evidence.
DRC_MAX_N_FRACTION = 0.20

# Transition-run requirement for STEP 1.
# Trimming requires: opposite run -> overlap/none -> reversed run.
DRC_MIN_OPPOSITE_RUN = 2
DRC_MIN_REVERSED_RUN = 2
DRC_REQUIRE_OVERLAP_STATE = True

# Debug printing is off by default for public use.
# The debug CSV can still be saved for traceability.
DRC_DEBUG = False
DRC_SAVE_DEBUG_REPORT = True

# =========================
# ITR detection settings
# =========================

ITR_SEARCH_WINDOW = 10000
MIN_ITR_LENGTH = 100
MIN_ITR_IDENTITY = 0.95
MIN_ITR_SCORE = 100
ITR_TERMINAL_TOLERANCE = 500

# Minimum number of informative A/C/G/T aligned positions.
# N or other ambiguous bases are excluded from identity and adjusted-score calculations.
MIN_ITR_INFORMATIVE_BASES = 100


In [5]:
import os
import re
import csv
from Bio import SeqIO, Align
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord


def make_safe_filename(text, max_len=180):
    """Convert a FASTA header into a safe file or folder name."""
    safe = re.sub(r'[\\/:*?"<>|]+', '_', str(text).strip())
    safe = re.sub(r'\s+', '_', safe)
    safe = re.sub(r'_+', '_', safe).strip('._ ')
    return safe[:max_len] if safe else 'sequence'


def _n_fraction(seq):
    """Return the fraction of non-ACGT bases in a sequence window."""
    seq = str(seq).upper()
    if not seq:
        return 1.0
    non_acgt = sum(1 for base in seq if base not in {"A", "C", "G", "T"})
    return non_acgt / len(seq)


# ============================================================
# STEP 1 helper: DRC boundary detection
# ============================================================

def find_drc_boundary_smart(
    seq_chunk,
    is_left=True,
    coarse_step=2000,
    fine_step=100,
    window_size=300,
    min_identity=0.85,
    min_alignment_coverage=0.80,
    margin=300,
    max_n_fraction=0.20,
    min_opposite_run=2,
    min_reversed_run=2,
    require_overlap_state=True,
    debug=False,
    return_details=True,
):
    """
    Detect a DRC trimming boundary from one terminal chunk.

    Phase 1 performs a coarse scan and brackets a candidate region.
    Phase 2 performs a fine scan and requires a structural transition:

        opposite run >= min_opposite_run
        -> overlap_or_none, if require_overlap_state is True
        -> reversed run >= min_reversed_run

    A single local-alignment hit is not enough to trigger trimming.
    Windows with too many ambiguous bases are ignored.
    """
    aligner = Align.PairwiseAligner()
    aligner.mode = 'local'
    aligner.match_score = 1
    aligner.mismatch_score = -1
    aligner.open_gap_score = -1
    aligner.extend_gap_score = -1

    seq_str = str(seq_chunk).upper()
    chunk_len = len(seq_str)
    min_score = window_size * min_identity

    details = {
        "side": "Left" if is_left else "Right",
        "phase1_candidate": False,
        "phase1_i": "",
        "phase1_j": "",
        "phase1_state": "",
        "phase1_score": "",
        "phase1_method": "",
        "phase1_coverage": "",
        "phase1_aligned_len": "",
        "phase1_n_fraction": "",
        "phase2_pass": False,
        "phase2_i": "",
        "phase2_j": "",
        "phase2_state": "",
        "phase2_score": "",
        "phase2_method": "",
        "phase2_coverage": "",
        "phase2_aligned_len": "",
        "phase2_n_fraction": "",
        "phase2_opposite_run": 0,
        "phase2_reversed_run": 0,
        "phase2_overlap_count": 0,
        "min_opposite_run": min_opposite_run,
        "min_reversed_run": min_reversed_run,
        "require_overlap_state": require_overlap_state,
        "max_n_fraction": max_n_fraction,
        "transition_opposite_i": "",
        "transition_overlap_i": "",
        "transition_reversed_i": "",
        "cut": "",
        "window": "",
        "rc_window": "",
        "note": "",
    }

    def _print_debug(message=None):
        if not debug:
            return
        side = details["side"]
        if message:
            print(f"      [DRC DEBUG - {side}] {message}")
        print(
            f"        Phase1 candidate={details['phase1_candidate']} "
            f"i={details['phase1_i']} j={details['phase1_j']} "
            f"state={details['phase1_state']} score={details['phase1_score']} "
            f"method={details['phase1_method']} coverage={details['phase1_coverage']} "
            f"aligned_len={details['phase1_aligned_len']} n_fraction={details['phase1_n_fraction']}"
        )
        print(
            f"        Phase2 pass={details['phase2_pass']} "
            f"i={details['phase2_i']} j={details['phase2_j']} "
            f"state={details['phase2_state']} score={details['phase2_score']} "
            f"method={details['phase2_method']} coverage={details['phase2_coverage']} "
            f"aligned_len={details['phase2_aligned_len']} n_fraction={details['phase2_n_fraction']} "
            f"cut={details['cut']}"
        )
        print(
            f"        transition_run: opposite_run={details['phase2_opposite_run']} "
            f"overlap_count={details['phase2_overlap_count']} "
            f"reversed_run={details['phase2_reversed_run']} "
            f"required={details['min_opposite_run']} -> overlap -> {details['min_reversed_run']}"
        )
        print(
            f"        transition_i: opposite={details['transition_opposite_i']} "
            f"overlap={details['transition_overlap_i']} "
            f"reversed={details['transition_reversed_i']}"
        )
        if details["window"]:
            print(f"        fwd_window: {details['window']}")
            print(f"        rc_window : {details['rc_window']}")

    def _find_rc_partner(fwd_window, rc_window):
        """
        Return partner position, score, method, coverage, aligned length, and N fraction.

        Exact hits are preferred. If no exact hit is found, local alignment is used.
        Local hits must satisfy both score and query-coverage thresholds.
        """
        n_frac = _n_fraction(fwd_window)
        if n_frac > max_n_fraction:
            return -1, 0.0, "skipped_high_N", 0.0, 0, n_frac

        j = seq_str.find(rc_window)
        if j != -1:
            return j, float(window_size), "exact", 1.0, window_size, n_frac

        try:
            best_aln = next(aligner.align(rc_window, seq_str))
            if len(best_aln.aligned[0]) > 0 and len(best_aln.aligned[1]) > 0:
                query_blocks = best_aln.aligned[0]
                aligned_len = int(sum(end - start for start, end in query_blocks))
                coverage = aligned_len / window_size if window_size else 0.0
                if best_aln.score >= min_score and coverage >= min_alignment_coverage:
                    return int(best_aln.aligned[1][0][0]), float(best_aln.score), "local", float(coverage), aligned_len, n_frac
        except StopIteration:
            pass

        return -1, 0.0, "none", 0.0, 0, n_frac

    def _classify_state(i, j, method):
        if method == "skipped_high_N":
            return "invalid_high_N"

        if is_left:
            is_opposite = (j != -1 and j > i + window_size)
            is_overlap_or_none = (j == -1 or abs(j - i) <= window_size)
            is_reversed = (j != -1 and j < i)
        else:
            is_opposite = (j != -1 and j < i)
            is_overlap_or_none = (j == -1 or abs(j - i) <= window_size)
            is_reversed = (j != -1 and j > i + window_size)

        if is_opposite:
            return "opposite"
        if is_overlap_or_none:
            return "overlap_or_none"
        if is_reversed:
            return "reversed"
        return "other"

    def _return(value):
        if return_details:
            return value, details
        return value

    if chunk_len < window_size:
        details["note"] = "Chunk_shorter_than_window"
        return _return(None)

    # Phase 1: coarse scan to bracket a candidate region.
    search_range_coarse = (
        range(0, chunk_len - window_size, coarse_step)
        if is_left else
        range(chunk_len - window_size, -1, -coarse_step)
    )

    refined_start = None
    seen_opposite = False

    for i in search_range_coarse:
        fwd_window = seq_str[i:i + window_size]
        rc_window = str(Seq(fwd_window).reverse_complement())
        j, score, method, coverage, aligned_len, n_frac = _find_rc_partner(fwd_window, rc_window)
        state = _classify_state(i, j, method)

        if state == "opposite":
            seen_opposite = True
        elif state in ("overlap_or_none", "reversed") and seen_opposite:
            details.update({
                "phase1_candidate": True,
                "phase1_i": i,
                "phase1_j": j,
                "phase1_state": state,
                "phase1_score": round(score, 2),
                "phase1_method": method,
                "phase1_coverage": round(coverage, 3),
                "phase1_aligned_len": aligned_len,
                "phase1_n_fraction": round(n_frac, 3),
            })
            refined_start = max(0, i - (coarse_step * 3)) if is_left else min(chunk_len - window_size, i + (coarse_step * 3))
            break

    if refined_start is None:
        details["note"] = "No_phase1_candidate"
        return _return(None)

    # Phase 2: fine scan with explicit transition-run requirement.
    scan_distance = coarse_step * 4
    fine_range = (
        range(refined_start, refined_start + scan_distance, fine_step)
        if is_left else
        range(refined_start, refined_start - scan_distance, -fine_step)
    )

    stage = "seeking_opposite_run"
    opposite_run = 0
    reversed_run = 0
    overlap_count = 0
    last_opposite_i = ""
    first_overlap_i = ""
    first_reversed_i = ""
    first_reversed_hit = None

    for i in fine_range:
        if i < 0 or i > chunk_len - window_size:
            continue

        fwd_window = seq_str[i:i + window_size]
        rc_window = str(Seq(fwd_window).reverse_complement())
        j, score, method, coverage, aligned_len, n_frac = _find_rc_partner(fwd_window, rc_window)
        state = _classify_state(i, j, method)
        current_hit = (i, j, state, score, method, coverage, aligned_len, n_frac, fwd_window, rc_window)

        if stage == "seeking_opposite_run":
            if state == "opposite":
                opposite_run += 1
                last_opposite_i = i
                if opposite_run >= min_opposite_run:
                    stage = "seeking_overlap"
            else:
                opposite_run = 0

        elif stage == "seeking_overlap":
            if state == "opposite":
                opposite_run += 1
                last_opposite_i = i
            elif state == "overlap_or_none":
                overlap_count += 1
                if first_overlap_i == "":
                    first_overlap_i = i
                stage = "seeking_reversed_run"
            elif state == "reversed" and not require_overlap_state:
                reversed_run = 1
                first_reversed_i = i
                first_reversed_hit = current_hit
                stage = "seeking_reversed_run"
            else:
                opposite_run = 0
                reversed_run = 0
                overlap_count = 0
                last_opposite_i = ""
                first_overlap_i = ""
                first_reversed_i = ""
                first_reversed_hit = None
                stage = "seeking_opposite_run"

        elif stage == "seeking_reversed_run":
            if state == "overlap_or_none":
                overlap_count += 1
            elif state == "reversed":
                reversed_run += 1
                if first_reversed_i == "":
                    first_reversed_i = i
                    first_reversed_hit = current_hit

                if reversed_run >= min_reversed_run:
                    cut_i, cut_j, cut_state, cut_score, cut_method, cut_cov, cut_aligned_len, cut_n_frac, cut_window, cut_rc_window = first_reversed_hit
                    cut = max(0, cut_i - margin) if is_left else min(chunk_len, cut_i + window_size + margin)
                    details.update({
                        "phase2_pass": True,
                        "phase2_i": cut_i,
                        "phase2_j": cut_j,
                        "phase2_state": cut_state,
                        "phase2_score": round(cut_score, 2),
                        "phase2_method": cut_method,
                        "phase2_coverage": round(cut_cov, 3),
                        "phase2_aligned_len": cut_aligned_len,
                        "phase2_n_fraction": round(cut_n_frac, 3),
                        "phase2_opposite_run": opposite_run,
                        "phase2_reversed_run": reversed_run,
                        "phase2_overlap_count": overlap_count,
                        "transition_opposite_i": last_opposite_i,
                        "transition_overlap_i": first_overlap_i,
                        "transition_reversed_i": first_reversed_i,
                        "cut": cut,
                        "window": cut_window,
                        "rc_window": cut_rc_window,
                        "note": "PASS_transition_run",
                    })
                    _print_debug("DRC trimming evidence: transition-run passed")
                    return _return(cut)
            else:
                opposite_run = 1 if state == "opposite" else 0
                reversed_run = 0
                overlap_count = 0
                last_opposite_i = i if state == "opposite" else ""
                first_overlap_i = ""
                first_reversed_i = ""
                first_reversed_hit = None
                stage = "seeking_overlap" if opposite_run >= min_opposite_run else "seeking_opposite_run"

    details.update({
        "phase2_opposite_run": opposite_run,
        "phase2_reversed_run": reversed_run,
        "phase2_overlap_count": overlap_count,
        "transition_opposite_i": last_opposite_i,
        "transition_overlap_i": first_overlap_i,
        "transition_reversed_i": first_reversed_i,
        "note": "Phase1_candidate_but_transition_run_failed",
    })
    _print_debug("Phase 1 found a candidate, but Phase 2 did not pass the transition-run rule.")
    return _return(None)


def _make_drc_debug_row(seq_id, fasta_header, side, info):
    """Convert a DRC details dictionary into one CSV row."""
    info = info or {}
    row = {
        "FASTA_Header": fasta_header,
        "Side": side,
        "Pass": info.get("phase2_pass", False),
        "Cut": info.get("cut", ""),
        "Note": info.get("note", ""),
        "Phase1_Candidate": info.get("phase1_candidate", ""),
        "Phase1_i": info.get("phase1_i", ""),
        "Phase1_j": info.get("phase1_j", ""),
        "Phase1_State": info.get("phase1_state", ""),
        "Phase1_Score": info.get("phase1_score", ""),
        "Phase1_Method": info.get("phase1_method", ""),
        "Phase1_Coverage": info.get("phase1_coverage", ""),
        "Phase1_Aligned_Len": info.get("phase1_aligned_len", ""),
        "Phase1_N_Fraction": info.get("phase1_n_fraction", ""),
        "Phase2_i": info.get("phase2_i", ""),
        "Phase2_j": info.get("phase2_j", ""),
        "Phase2_State": info.get("phase2_state", ""),
        "Phase2_Score": info.get("phase2_score", ""),
        "Phase2_Method": info.get("phase2_method", ""),
        "Phase2_Coverage": info.get("phase2_coverage", ""),
        "Phase2_Aligned_Len": info.get("phase2_aligned_len", ""),
        "Phase2_N_Fraction": info.get("phase2_n_fraction", ""),
        "Opposite_Run": info.get("phase2_opposite_run", ""),
        "Overlap_Count": info.get("phase2_overlap_count", ""),
        "Reversed_Run": info.get("phase2_reversed_run", ""),
        "Required_Opposite_Run": info.get("min_opposite_run", ""),
        "Required_Reversed_Run": info.get("min_reversed_run", ""),
        "Require_Overlap_State": info.get("require_overlap_state", ""),
        "Max_N_Fraction": info.get("max_n_fraction", ""),
        "Transition_Opposite_i": info.get("transition_opposite_i", ""),
        "Transition_Overlap_i": info.get("transition_overlap_i", ""),
        "Transition_Reversed_i": info.get("transition_reversed_i", ""),
        "Window": info.get("window", ""),
        "RC_Window": info.get("rc_window", ""),
    }
    return row


# ============================================================
# STEP 2 helpers: hairpin and palindrome refinement
# ============================================================

def find_hairpin_sliding_window(seq_str, search_limit=1000, window_size=20, min_identity=0.9):
    """
    Detect a terminal hairpin-like stem using a sliding reverse-complement window.

    A perfect 20-bp match extends the safe stem boundary.
    A one-mismatch window is tolerated as a transient error.
    A lower-scoring window is treated as loop entry, and the function returns the last safe boundary.
    """
    search_region = seq_str[:search_limit]

    aligner = Align.PairwiseAligner()
    aligner.mode = 'local'
    aligner.match_score = 1
    aligner.mismatch_score = -1
    aligner.open_gap_score = -1
    aligner.extend_gap_score = -1

    final_stem_end = 0
    final_rc_start = -1
    error_start_i = None
    last_perfect_rc_start = -1

    for i in range(window_size, len(search_region) - window_size):
        fwd_chunk = search_region[i - window_size:i]
        rc_chunk = str(Seq(fwd_chunk).reverse_complement())
        target = search_region[i:i + 1000]

        try:
            best_aln = next(aligner.align(rc_chunk, target))
            score = 0 if len(best_aln.aligned[0]) == 0 or len(best_aln.aligned[1]) == 0 else best_aln.score
        except StopIteration:
            score = 0

        identity = score / window_size

        if score == window_size:
            error_start_i = None
            last_perfect_rc_start = i + best_aln.aligned[1][0][0]
        elif identity >= min_identity:
            if error_start_i is None:
                error_start_i = i
        else:
            final_stem_end = (error_start_i - 1) if error_start_i is not None else (i - 1)
            final_rc_start = last_perfect_rc_start
            break

    loop_seq = ""
    trimmed_seq = seq_str

    if final_stem_end > 0 and final_rc_start > final_stem_end:
        loop_seq = search_region[final_stem_end:final_rc_start]
        trimmed_seq = seq_str[final_stem_end:]

    return final_stem_end, final_rc_start, loop_seq, trimmed_seq


def find_palindrome_center_fast(seq_str, search_limit=1000, min_len=40):
    """
    Fallback palindrome detector for STEP 2.

    The allowed start window is relaxed because STEP 1 intentionally leaves a margin.
    """
    search_region = seq_str[:search_limit]
    rc_region = str(Seq(search_region).reverse_complement())

    aligner = Align.PairwiseAligner()
    aligner.mode = 'local'
    aligner.match_score = 2
    aligner.mismatch_score = -3
    aligner.open_gap_score = -5
    aligner.extend_gap_score = -2

    try:
        alignments = aligner.align(search_region, rc_region)
        for aln in alignments:
            start = aln.aligned[0][0][0]
            end = aln.aligned[0][-1][1]
            match_len = end - start
            if start < 500 and match_len >= min_len:
                return int((start + end) // 2)
    except StopIteration:
        pass

    return 0


# ============================================================
# STEP 3 helpers: ITR detection
# ============================================================

def _alignment_identity_from_blocks(left_seq, right_seq, left_blocks, right_blocks):
    """
    Calculate identity over ungapped aligned blocks.

    Positions containing N or any non-ACGT base are excluded from match, mismatch,
    identity, and adjusted-score calculations. Coordinates are not changed.
    """
    valid_bases = {"A", "C", "G", "T"}
    matches = 0
    compared = 0
    ambiguous_skipped = 0
    adjusted_score = 0.0

    for (l_start, l_end), (r_start, r_end) in zip(left_blocks, right_blocks):
        block_len = min(int(l_end) - int(l_start), int(r_end) - int(r_start))
        if block_len <= 0:
            continue

        left_block = left_seq[int(l_start):int(l_start) + block_len]
        right_block = right_seq[int(r_start):int(r_start) + block_len]

        for a, b in zip(left_block, right_block):
            if a not in valid_bases or b not in valid_bases:
                ambiguous_skipped += 1
                continue
            compared += 1
            if a == b:
                matches += 1
                adjusted_score += 2
            else:
                adjusted_score -= 3

    identity = (matches / compared) if compared else 0.0
    return identity, matches, compared, ambiguous_skipped, adjusted_score


def find_dual_itr_alignment_safe(
    left_terminal,
    right_terminal_rc,
    min_itr_length=100,
    min_identity=0.80,
    min_score=100,
    terminal_tolerance=50,
    min_informative_bases=100,
):
    """
    Detect an ITR candidate by aligning left terminal sequence against the
    reverse-complemented right terminal sequence.

    A candidate must pass length, identity, adjusted-score, terminal-proximity,
    and informative-base filters. N and other ambiguous bases are excluded from
    identity and adjusted-score calculations.
    """
    result = {
        "found": False,
        "status": "No_alignment",
        "left_start": 0,
        "left_end": 0,
        "right_rc_start": 0,
        "right_rc_end": 0,
        "left_len": 0,
        "right_len": 0,
        "len_diff": 0,
        "len_diff_ratio": 1.0,
        "identity": 0.0,
        "score": 0.0,
        "raw_score": 0.0,
        "matches": 0,
        "aligned_bases": 0,
        "ambiguous_skipped": 0,
        "length_ok": False,
        "identity_ok": False,
        "score_ok": False,
        "terminal_ok": False,
        "informative_bases_ok": False,
        "fail_reasons": "No_alignment",
    }

    left_seq = str(left_terminal).upper()
    right_seq = str(right_terminal_rc).upper()

    if len(left_seq) == 0 or len(right_seq) == 0:
        result["status"] = "No_sequence"
        result["fail_reasons"] = "No_sequence"
        return result

    aligner = Align.PairwiseAligner()
    aligner.mode = 'local'
    aligner.match_score = 2
    aligner.mismatch_score = -3
    aligner.open_gap_score = -5
    aligner.extend_gap_score = -2

    try:
        best = next(aligner.align(left_seq, right_seq))
        result["raw_score"] = float(best.score)
        l_idx, r_idx = best.aligned[0], best.aligned[1]

        if len(l_idx) == 0 or len(r_idx) == 0:
            return result

        left_start = int(l_idx[0][0])
        left_end = int(l_idx[-1][1])
        right_start = int(r_idx[0][0])
        right_end = int(r_idx[-1][1])

        left_len = max(0, left_end - left_start)
        right_len = max(0, right_end - right_start)
        len_diff = abs(left_len - right_len)
        longer_len = max(left_len, right_len)
        len_diff_ratio = (len_diff / longer_len) if longer_len else 1.0

        identity, matches, aligned_bases, ambiguous_skipped, adjusted_score = _alignment_identity_from_blocks(
            left_seq, right_seq, l_idx, r_idx
        )

        result.update({
            "left_start": left_start,
            "left_end": left_end,
            "right_rc_start": right_start,
            "right_rc_end": right_end,
            "left_len": left_len,
            "right_len": right_len,
            "len_diff": len_diff,
            "len_diff_ratio": len_diff_ratio,
            "identity": identity,
            "score": adjusted_score,
            "matches": matches,
            "aligned_bases": aligned_bases,
            "ambiguous_skipped": ambiguous_skipped,
        })

        result["length_ok"] = (left_len >= min_itr_length and right_len >= min_itr_length)
        result["identity_ok"] = (identity >= min_identity)
        result["score_ok"] = (adjusted_score >= min_score)
        result["terminal_ok"] = (left_start <= terminal_tolerance or right_start <= terminal_tolerance)
        result["informative_bases_ok"] = (aligned_bases >= min_informative_bases)

        fail_reasons = []
        if not result["length_ok"]:
            fail_reasons.append(f"length<{min_itr_length}")
        if not result["identity_ok"]:
            fail_reasons.append(f"identity<{min_identity}")
        if not result["score_ok"]:
            fail_reasons.append(f"adjusted_score<{min_score}")
        if not result["terminal_ok"]:
            fail_reasons.append(f"neither_start_terminal<={terminal_tolerance}")
        if not result["informative_bases_ok"]:
            fail_reasons.append(f"informative_bases<{min_informative_bases}")

        if not fail_reasons:
            result["found"] = True
            result["status"] = "Found"
            result["fail_reasons"] = "PASS"
        else:
            result["status"] = "Rejected"
            result["fail_reasons"] = ";".join(fail_reasons)

        return result

    except Exception as e:
        result["status"] = "Error"
        result["fail_reasons"] = str(e)
        return result


# ============================================================
# STEP 2.5 helpers: reference-based orientation normalization
# ============================================================

def load_reference_sequence(reference_fasta):
    """Load the first record from a reference genome FASTA file."""
    if reference_fasta is None or str(reference_fasta).strip() == "":
        return None, ""

    if not os.path.exists(reference_fasta):
        raise FileNotFoundError(f"Reference FASTA file was not found: {reference_fasta}")

    ref_records = list(SeqIO.parse(reference_fasta, "fasta"))
    if len(ref_records) == 0:
        raise ValueError(f"No FASTA records were found in the reference file: {reference_fasta}")

    return ref_records[0].seq, ref_records[0].description


def build_reference_kmer_set(reference_seq, k=31):
    """Build a reference k-mer set using only A/C/G/T k-mers."""
    ref_str = str(reference_seq).upper()
    if len(ref_str) < k:
        return {ref_str} if re.fullmatch(r"[ACGT]+", ref_str) else set()

    return {
        ref_str[i:i + k]
        for i in range(0, len(ref_str) - k + 1)
        if re.fullmatch(r"[ACGT]+", ref_str[i:i + k])
    }


def sample_sequence_chunks(seq_str, chunk_size=10000, n_chunks=25):
    """Sample evenly spaced chunks from a sequence for fast k-mer scoring."""
    seq_str = str(seq_str).upper()
    seq_len = len(seq_str)

    if seq_len == 0:
        return []
    if seq_len <= chunk_size:
        return [seq_str]
    if n_chunks <= 1:
        return [seq_str[:chunk_size]]

    max_start = seq_len - chunk_size
    starts = sorted({round(i * max_start / (n_chunks - 1)) for i in range(n_chunks)})
    return [seq_str[start:start + chunk_size] for start in starts]


def score_sequence_against_reference_kmers(seq, reference_kmers, k=21, chunk_size=10000, n_chunks=25, step=1):
    """Count sampled query k-mers that are present in the reference k-mer set."""
    if not reference_kmers:
        return 0, 0, 0.0

    matched = 0
    total = 0

    for chunk in sample_sequence_chunks(str(seq), chunk_size=chunk_size, n_chunks=n_chunks):
        if len(chunk) < k:
            continue
        for i in range(0, len(chunk) - k + 1, step):
            kmer = chunk[i:i + k]
            if not re.fullmatch(r"[ACGT]+", kmer):
                continue
            total += 1
            if kmer in reference_kmers:
                matched += 1

    ratio = matched / total if total else 0.0
    return matched, total, ratio


def orient_sequence_to_reference(seq, reference_kmers, k=21, chunk_size=10000, n_chunks=25, step=1, min_ratio=1.2, min_kmer_delta=5):
    """Orient a sequence by comparing forward and reverse-complement k-mer scores."""
    fwd_matched, fwd_total, fwd_ratio = score_sequence_against_reference_kmers(
        seq, reference_kmers, k=k, chunk_size=chunk_size, n_chunks=n_chunks, step=step
    )

    rc_seq = Seq(str(seq)).reverse_complement()
    rc_matched, rc_total, rc_ratio = score_sequence_against_reference_kmers(
        rc_seq, reference_kmers, k=k, chunk_size=chunk_size, n_chunks=n_chunks, step=step
    )

    if fwd_matched == 0 and rc_matched == 0:
        decision_ratio = 1.0
    else:
        smaller = min(fwd_matched, rc_matched)
        larger = max(fwd_matched, rc_matched)
        decision_ratio = float("inf") if smaller == 0 else larger / smaller

    rc_is_stronger = (rc_matched >= fwd_matched * min_ratio and rc_matched >= fwd_matched + min_kmer_delta)
    fwd_is_stronger = (fwd_matched >= rc_matched * min_ratio and fwd_matched >= rc_matched + min_kmer_delta)

    if rc_is_stronger:
        orientation = "reverse_complement"
        oriented_seq = rc_seq
    elif fwd_is_stronger:
        orientation = "forward"
        oriented_seq = Seq(str(seq))
    else:
        orientation = "ambiguous_keep_original"
        oriented_seq = Seq(str(seq))

    return oriented_seq, {
        "orientation": orientation,
        "forward_matched_kmers": fwd_matched,
        "forward_total_kmers": fwd_total,
        "forward_match_ratio": fwd_ratio,
        "reverse_matched_kmers": rc_matched,
        "reverse_total_kmers": rc_total,
        "reverse_match_ratio": rc_ratio,
        "decision_ratio": decision_ratio,
        "k": k,
        "chunk_size": chunk_size,
        "n_chunks": n_chunks,
        "step": step,
        "min_ratio": min_ratio,
        "min_kmer_delta": min_kmer_delta,
    }


# ============================================================
# Main pipeline
# ============================================================

def run_optimized_batch_pipeline(
    input_fasta,
    output_root="hairpin_finder_results",
    reference_fasta=None,
    orientation_k=15,
    orientation_chunk_size=10000,
    orientation_n_chunks=25,
    orientation_step=1,
    orientation_min_ratio=1.2,
    drc_coarse_step=2000,
    drc_fine_step=100,
    drc_window_size=300,
    drc_min_identity=0.85,
    drc_min_alignment_coverage=0.80,
    drc_margin=300,
    drc_max_n_fraction=0.20,
    drc_min_opposite_run=2,
    drc_min_reversed_run=2,
    drc_require_overlap_state=True,
    drc_debug=False,
    drc_save_debug_report=True,
    itr_search_window=10000,
    min_itr_length=100,
    min_itr_identity=0.80,
    min_itr_score=100,
    itr_terminal_tolerance=500,
    min_itr_informative_bases=100,
):
    """Run the complete terminal-structure trimming and reporting pipeline."""
    if not os.path.exists(input_fasta):
        raise FileNotFoundError(
            f"Input FASTA file was not found: {input_fasta}\n"
            "Check INPUT_FASTA in the configuration cell."
        )

    os.makedirs(output_root, exist_ok=True)

    records = list(SeqIO.parse(input_fasta, "fasta"))
    if len(records) == 0:
        raise ValueError(f"No FASTA records were found in the input file: {input_fasta}")

    reference_seq, reference_header = load_reference_sequence(reference_fasta)
    if reference_seq is not None:
        reference_kmers = build_reference_kmer_set(reference_seq, k=orientation_k)
        print(f"Reference genome loaded: {reference_header} ({len(reference_seq)} bp)")
        print(f"Reference k-mers: {len(reference_kmers)} (k={orientation_k})")
        print(
            "Orientation settings: "
            f"chunk_size={orientation_chunk_size}, "
            f"n_chunks={orientation_n_chunks}, "
            f"step={orientation_step}, "
            f"min_ratio={orientation_min_ratio}"
        )
    else:
        reference_kmers = None
        reference_header = ""

    summary_data = []
    drc_debug_rows = []
    all_final_records = []
    used_safe_names = {}

    print(f"Starting analysis for {len(records)} sequence(s)")
    print("=" * 50)

    for record in records:
        seq_header = record.description
        seq_id = record.id.replace("|", "_").replace("/", "_").replace(" ", "_")
        safe_header = make_safe_filename(seq_header)

        used_safe_names[safe_header] = used_safe_names.get(safe_header, 0) + 1
        safe_header_unique = f"{safe_header}_{used_safe_names[safe_header]}" if used_safe_names[safe_header] > 1 else safe_header

        full_seq = record.seq
        orig_len = len(full_seq)
        seq_folder = os.path.join(output_root, safe_header_unique)
        os.makedirs(seq_folder, exist_ok=True)

        print(f"\n> Starting analysis: {seq_header} (original length: {orig_len} bp)")

        l_cut = 0
        r_cut = orig_len
        status_tags = []
        final_seq = full_seq
        l_loop_seq_str = ""
        r_loop_seq_str = ""
        search_limit = ""
        error_message = ""

        orientation_result = "Not_checked"
        orientation_rc_applied = "Not_checked"
        orientation_action = "Not_checked"
        orientation_fwd_kmers = ""
        orientation_rc_kmers = ""
        orientation_fwd_ratio = ""
        orientation_rc_ratio = ""
        orientation_decision_ratio = ""
        orientation_min_ratio_used = orientation_min_ratio
        orientation_min_delta_used = ""

        itr_status = "Not_checked"
        itr_identity = ""
        itr_score = ""
        itr_raw_score = ""
        itr_matches = ""
        itr_aligned_bases = ""
        itr_ambiguous_skipped = ""
        itr_len_diff_ratio = ""
        itr_length_ok = ""
        itr_identity_ok = ""
        itr_score_ok = ""
        itr_terminal_ok = ""
        itr_informative_bases_ok = ""
        itr_fail_reasons = ""
        l_s_itr = l_e_itr = r_s_itr = r_e_itr = 0
        left_itr_len = 0
        right_itr_len = 0

        l_drc_pass = False
        r_drc_pass = False
        l_drc_score = ""
        r_drc_score = ""
        l_drc_coverage = ""
        r_drc_coverage = ""
        l_drc_note = ""
        r_drc_note = ""

        # STEP 1 DRC cuts and STEP 2 terminal-refinement cuts are tracked separately.
        # Summary columns DRC_L_Cut and DRC_R_Cut report the combined terminal cut length.
        l_step1_drc_cut_bp = 0
        r_step1_drc_cut_bp = 0
        l_step2_refine_cut_bp = 0
        r_step2_refine_cut_bp = 0

        try:
            print("  [STEP 1 - Smart DRC trimming]")
            search_limit = max(1, orig_len // 2)
            print(f"    - DRC search range: {search_limit} bp from each terminal (one half of the sequence)")

            left_chunk = full_seq[:search_limit]
            right_chunk = full_seq[-search_limit:]

            l_result, l_drc_info = find_drc_boundary_smart(
                left_chunk,
                is_left=True,
                coarse_step=drc_coarse_step,
                fine_step=drc_fine_step,
                window_size=drc_window_size,
                min_identity=drc_min_identity,
                min_alignment_coverage=drc_min_alignment_coverage,
                margin=drc_margin,
                max_n_fraction=drc_max_n_fraction,
                min_opposite_run=drc_min_opposite_run,
                min_reversed_run=drc_min_reversed_run,
                require_overlap_state=drc_require_overlap_state,
                debug=drc_debug,
            )
            drc_debug_rows.append(_make_drc_debug_row(seq_id, seq_header, "Left", l_drc_info))
            l_drc_pass = bool(l_drc_info.get("phase2_pass", False))
            l_drc_score = l_drc_info.get("phase2_score", "")
            l_drc_coverage = l_drc_info.get("phase2_coverage", "")
            l_drc_note = l_drc_info.get("note", "")

            if l_result is not None:
                l_cut = l_result
                l_step1_drc_cut_bp = l_cut
                status_tags.append("L_DRC")
                print(f"    - Left DRC cut: {l_cut}")
            else:
                print("    - No left DRC cut")

            r_result_relative, r_drc_info = find_drc_boundary_smart(
                right_chunk,
                is_left=False,
                coarse_step=drc_coarse_step,
                fine_step=drc_fine_step,
                window_size=drc_window_size,
                min_identity=drc_min_identity,
                min_alignment_coverage=drc_min_alignment_coverage,
                margin=drc_margin,
                max_n_fraction=drc_max_n_fraction,
                min_opposite_run=drc_min_opposite_run,
                min_reversed_run=drc_min_reversed_run,
                require_overlap_state=drc_require_overlap_state,
                debug=drc_debug,
            )
            drc_debug_rows.append(_make_drc_debug_row(seq_id, seq_header, "Right", r_drc_info))
            r_drc_pass = bool(r_drc_info.get("phase2_pass", False))
            r_drc_score = r_drc_info.get("phase2_score", "")
            r_drc_coverage = r_drc_info.get("phase2_coverage", "")
            r_drc_note = r_drc_info.get("note", "")

            if r_result_relative is not None:
                r_cut = (orig_len - search_limit) + r_result_relative
                r_step1_drc_cut_bp = orig_len - r_cut
                status_tags.append("R_DRC")
                print(f"    - Right DRC cut: {r_cut}")
            else:
                print("    - No right DRC cut")

            if l_cut >= r_cut:
                raise ValueError(f"Invalid trimming coordinates: l_cut={l_cut}, r_cut={r_cut}")

            step1_seq = full_seq[l_cut:r_cut]
            print(f"    > Length after STEP 1 trimming: {len(step1_seq)} bp")

            print("  [STEP 2 - Hairpin/palindrome refinement]")
            l_stem_len, l_rc_start, l_loop_seq_str, mid_seq_str = find_hairpin_sliding_window(str(step1_seq))

            if len(l_loop_seq_str) > 0:
                mid_seq = Seq(mid_seq_str)
                l_step2_refine_cut_bp = l_stem_len
                status_tags.append("L_Hairpin")
                print("    - Left hairpin-like structure detected")
                print(f"      * Removed pseudo-stem length: {l_stem_len} bp")
                print(f"      * Hairpin loop length: {len(l_loop_seq_str)} bp")
            else:
                if l_cut > 0:
                    center = find_palindrome_center_fast(str(step1_seq))
                    if center > 0:
                        mid_seq = Seq(str(step1_seq)[center:])
                        l_step2_refine_cut_bp = center
                        status_tags.append("L_Palindrome")
                        print("    - Left palindrome-like structure detected")
                        print(f"      * Palindrome center cut: {center} bp")
                    else:
                        mid_seq = Seq(mid_seq_str)
                        print("    - No left hairpin/palindrome refinement")
                else:
                    mid_seq = Seq(mid_seq_str)
                    print("    - No left terminal refinement")

            mid_seq_rev_str = str(mid_seq.reverse_complement())
            r_stem_len, r_rc_start, r_loop_seq_str, final_seq_rev_str = find_hairpin_sliding_window(mid_seq_rev_str)

            if len(r_loop_seq_str) > 0:
                final_seq = Seq(final_seq_rev_str).reverse_complement()
                r_loop_seq_str = str(Seq(r_loop_seq_str).reverse_complement())
                r_step2_refine_cut_bp = r_stem_len
                status_tags.append("R_Hairpin")
                print("    - Right hairpin-like structure detected")
                print(f"      * Removed pseudo-stem length: {r_stem_len} bp")
                print(f"      * Hairpin loop length: {len(r_loop_seq_str)} bp")
            else:
                if r_cut < orig_len:
                    center = find_palindrome_center_fast(mid_seq_rev_str)
                    if center > 0:
                        final_seq_rev_str = mid_seq_rev_str[center:]
                        final_seq = Seq(final_seq_rev_str).reverse_complement()
                        r_step2_refine_cut_bp = center
                        status_tags.append("R_Palindrome")
                        print("    - Right palindrome-like structure detected")
                        print(f"      * Palindrome center cut: {center} bp")
                    else:
                        final_seq = Seq(final_seq_rev_str).reverse_complement()
                        print("    - No right hairpin/palindrome refinement")
                else:
                    final_seq = Seq(final_seq_rev_str).reverse_complement()
                    print("    - No right terminal refinement")

            final_len = len(final_seq)
            print(f"    > Final length after STEP 2 refinement: {final_len} bp")

            if reference_kmers is not None:
                print("  [STEP 2.5 - Reference-based orientation normalization]")
                final_seq, orientation_info = orient_sequence_to_reference(
                    final_seq,
                    reference_kmers,
                    k=orientation_k,
                    chunk_size=orientation_chunk_size,
                    n_chunks=orientation_n_chunks,
                    step=orientation_step,
                    min_ratio=orientation_min_ratio,
                )

                orientation_result = orientation_info["orientation"]
                orientation_fwd_kmers = orientation_info["forward_matched_kmers"]
                orientation_rc_kmers = orientation_info["reverse_matched_kmers"]
                orientation_fwd_ratio = round(orientation_info["forward_match_ratio"], 4)
                orientation_rc_ratio = round(orientation_info["reverse_match_ratio"], 4)
                orientation_decision_ratio = orientation_info["decision_ratio"]
                orientation_min_ratio_used = orientation_info["min_ratio"]
                orientation_min_delta_used = orientation_info["min_kmer_delta"]
                decision_ratio_print = "inf" if orientation_decision_ratio == float("inf") else round(orientation_decision_ratio, 4)

                if orientation_result == "reverse_complement":
                    orientation_rc_applied = "Yes"
                    orientation_action = "Reverse-complemented"
                    status_tags.append("Orientation_RC")
                    print("    - Orientation changed to reverse-complement")
                elif orientation_result == "forward":
                    orientation_rc_applied = "No"
                    orientation_action = "Forward retained"
                    status_tags.append("Orientation_FWD")
                    print("    - Forward orientation retained")
                else:
                    orientation_rc_applied = "No"
                    orientation_action = "Original retained (ambiguous)"
                    status_tags.append("Orientation_Ambiguous")
                    print("    - Orientation score difference was below threshold; original orientation retained")

                print(f"      * Forward matched k-mers: {orientation_fwd_kmers} (ratio={orientation_fwd_ratio})")
                print(f"      * Reverse-complement matched k-mers: {orientation_rc_kmers} (ratio={orientation_rc_ratio})")
                print(f"      * Decision ratio: {decision_ratio_print} (required >= {orientation_min_ratio_used}; min delta={orientation_min_delta_used})")
                final_len = len(final_seq)
            else:
                orientation_result = "Skipped_no_reference"
                orientation_rc_applied = "Skipped"
                orientation_action = "Skipped (no reference)"

            print("  [STEP 3 - ITR detection]")
            itr_window = min(itr_search_window, final_len)
            itr_result = find_dual_itr_alignment_safe(
                final_seq[:itr_window],
                final_seq[-itr_window:].reverse_complement(),
                min_itr_length=min_itr_length,
                min_identity=min_itr_identity,
                min_score=min_itr_score,
                terminal_tolerance=itr_terminal_tolerance,
                min_informative_bases=min_itr_informative_bases,
            )

            l_s_itr = itr_result["left_start"]
            l_e_itr = itr_result["left_end"]
            r_s_itr = itr_result["right_rc_start"]
            r_e_itr = itr_result["right_rc_end"]
            left_itr_len = itr_result["left_len"]
            right_itr_len = itr_result["right_len"]
            itr_status = itr_result["status"]
            itr_identity = round(itr_result["identity"], 4)
            itr_score = round(itr_result["score"], 2)
            itr_raw_score = round(itr_result["raw_score"], 2)
            itr_matches = itr_result["matches"]
            itr_aligned_bases = itr_result["aligned_bases"]
            itr_ambiguous_skipped = itr_result["ambiguous_skipped"]
            itr_len_diff_ratio = round(itr_result["len_diff_ratio"], 4)
            itr_length_ok = itr_result["length_ok"]
            itr_identity_ok = itr_result["identity_ok"]
            itr_score_ok = itr_result["score_ok"]
            itr_terminal_ok = itr_result["terminal_ok"]
            itr_informative_bases_ok = itr_result["informative_bases_ok"]
            itr_fail_reasons = itr_result["fail_reasons"]

            if itr_result["found"]:
                status_tags.append("ITR_Found")

            print(f"    - ITR decision: {itr_status} ({itr_fail_reasons})")
            print(f"    - Left ITR candidate: {l_s_itr} ~ {l_e_itr} (len={left_itr_len})")
            print(f"    - Right ITR candidate on RC terminal: {r_s_itr} ~ {r_e_itr} (len={right_itr_len})")
            print(
                f"    - identity={itr_identity}, adjusted_score={itr_score}, raw_score={itr_raw_score}, "
                f"informative_bases={itr_aligned_bases}, N/ambiguous_skipped={itr_ambiguous_skipped}, "
                f"len_diff_ratio={itr_len_diff_ratio}"
            )

            status_str = "|".join(status_tags) if status_tags else "Clean"
            detail_desc = f"{seq_header} | orig_len={orig_len} final_len={final_len} status={status_str}"
            final_record = SeqRecord(final_seq, id=record.id, description=detail_desc)

            final_fasta = os.path.join(seq_folder, f"{safe_header_unique}.fasta")
            SeqIO.write(final_record, final_fasta, "fasta")
            all_final_records.append(final_record)

            print(f"Completed: {seq_header} (final length: {final_len} bp | status: {status_str})")

        except Exception as e:
            final_len = 0
            left_itr_len = 0
            right_itr_len = 0
            status_str = "Error"
            error_message = str(e)
            print(f"Error while processing {seq_header}: {error_message}")

        total_left_terminal_cut_bp = l_step1_drc_cut_bp + l_step2_refine_cut_bp
        total_right_terminal_cut_bp = r_step1_drc_cut_bp + r_step2_refine_cut_bp

        summary_data.append({
            "FASTA_Header": seq_header,
            "Status": status_str,
            "Original_Length": orig_len,
            "DRC_Search_Limit": search_limit,
            "L_DRC_Pass": l_drc_pass,
            "R_DRC_Pass": r_drc_pass,

            # Total terminal cut lengths: STEP 1 DRC cut + STEP 2 hairpin/palindrome refinement cut.
            "DRC_L_Cut": total_left_terminal_cut_bp,
            "DRC_R_Cut": total_right_terminal_cut_bp,

            # Separate components are retained for traceability.
            "L_STEP1_DRC_Cut_Bp": l_step1_drc_cut_bp,
            "R_STEP1_DRC_Cut_Bp": r_step1_drc_cut_bp,
            "L_STEP2_Refinement_Cut_Bp": l_step2_refine_cut_bp,
            "R_STEP2_Refinement_Cut_Bp": r_step2_refine_cut_bp,

            # Original coordinate-style cut values are retained for debugging.
            "L_DRC_Cut_Coord": l_cut if l_drc_pass else 0,
            "R_DRC_Cut_Coord": r_cut if r_drc_pass else 0,
            "L_DRC_Score": l_drc_score,
            "L_DRC_Coverage": l_drc_coverage,
            "L_DRC_Note": l_drc_note,
            "R_DRC_Score": r_drc_score,
            "R_DRC_Coverage": r_drc_coverage,
            "R_DRC_Note": r_drc_note,
            "DRC_Coarse_Step": drc_coarse_step,
            "DRC_Fine_Step": drc_fine_step,
            "DRC_Window_Size": drc_window_size,
            "DRC_Min_Identity": drc_min_identity,
            "DRC_Min_Alignment_Coverage": drc_min_alignment_coverage,
            "DRC_Margin": drc_margin,
            "DRC_Max_N_Fraction": drc_max_n_fraction,
            "DRC_Min_Opposite_Run": drc_min_opposite_run,
            "DRC_Min_Reversed_Run": drc_min_reversed_run,
            "DRC_Require_Overlap_State": drc_require_overlap_state,
            "Reference_Header": reference_header,
            "Orientation": orientation_result,
            "Orientation_RC_Applied": orientation_rc_applied,
            "Orientation_Action": orientation_action,
            "Orientation_FWD_Kmers": orientation_fwd_kmers,
            "Orientation_RC_Kmers": orientation_rc_kmers,
            "Orientation_FWD_Ratio": orientation_fwd_ratio,
            "Orientation_RC_Ratio": orientation_rc_ratio,
            "Orientation_Decision_Ratio": orientation_decision_ratio,
            "Orientation_K": orientation_k,
            "Orientation_Chunk_Size": orientation_chunk_size,
            "Orientation_N_Chunks": orientation_n_chunks,
            "Orientation_Step": orientation_step,
            "Orientation_Min_Ratio": orientation_min_ratio_used,
            "Orientation_Min_Delta": orientation_min_delta_used,
            "Final_Length": final_len,
            "Trimmed_Bp": (orig_len - final_len) if final_len else "",
            "L_Cut": l_cut,
            "R_Cut": r_cut,
            "L_Loop_Len": len(l_loop_seq_str),
            "L_Loop_Seq": l_loop_seq_str,
            "R_Loop_Len": len(r_loop_seq_str),
            "R_Loop_Seq": r_loop_seq_str,
            "Candidate_L_ITR_Start": l_s_itr if left_itr_len else 0,
            "Candidate_L_ITR_End": l_e_itr if left_itr_len else 0,
            "Candidate_L_ITR_Len": left_itr_len,
            "Candidate_R_ITR_Start": (final_len - r_e_itr) if right_itr_len else 0,
            "Candidate_R_ITR_End": (final_len - r_s_itr) if right_itr_len else 0,
            "Candidate_R_ITR_Len": right_itr_len,
            "L_ITR_Start": l_s_itr if itr_status == "Found" else 0,
            "L_ITR_End": l_e_itr if itr_status == "Found" else 0,
            "L_ITR_Len": left_itr_len if itr_status == "Found" else 0,
            "R_ITR_Start": (final_len - r_e_itr) if itr_status == "Found" else 0,
            "R_ITR_End": (final_len - r_s_itr) if itr_status == "Found" else 0,
            "R_ITR_Len": right_itr_len if itr_status == "Found" else 0,
            "ITR_Len_Diff": abs(left_itr_len - right_itr_len) if itr_status == "Found" else 0,
            "ITR_Pass": itr_status == "Found",
            "ITR_Status": itr_status,
            "ITR_Identity": itr_identity,
            "ITR_Score": itr_score,
            "ITR_Raw_Score": itr_raw_score,
            "ITR_Matches": itr_matches,
            "ITR_Informative_Bases": itr_aligned_bases,
            "ITR_N_or_Ambiguous_Skipped": itr_ambiguous_skipped,
            "ITR_Len_Diff_Ratio": itr_len_diff_ratio,
            "ITR_Length_OK": itr_length_ok,
            "ITR_Identity_OK": itr_identity_ok,
            "ITR_Score_OK": itr_score_ok,
            "ITR_Terminal_OK": itr_terminal_ok,
            "ITR_Informative_Bases_OK": itr_informative_bases_ok,
            "ITR_Fail_Reasons": itr_fail_reasons,
            "ITR_Search_Window": itr_search_window,
            "ITR_Min_Length": min_itr_length,
            "ITR_Min_Identity": min_itr_identity,
            "ITR_Min_Score": min_itr_score,
            "ITR_Terminal_Tolerance": itr_terminal_tolerance,
            "ITR_Min_Informative_Bases": min_itr_informative_bases,
            "Error": error_message,
        })

    summary_file = os.path.join(output_root, "summary_report.csv")
    fieldnames = list(summary_data[0].keys()) if summary_data else []
    with open(summary_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(summary_data)

    drc_debug_file = ""
    if drc_save_debug_report and drc_debug_rows:
        drc_debug_file = os.path.join(output_root, "drc_debug_report.csv")
        debug_fieldnames = list(drc_debug_rows[0].keys())
        with open(drc_debug_file, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=debug_fieldnames)
            writer.writeheader()
            writer.writerows(drc_debug_rows)

    combined_fasta = os.path.join(output_root, "all_final_sequences.fasta")
    if len(all_final_records) > 0:
        SeqIO.write(all_final_records, combined_fasta, "fasta")
    else:
        combined_fasta = ""

    print("\nAll analyses completed.")
    print(f"   - Summary CSV: {summary_file}")
    if drc_debug_file:
        print(f"   - DRC debug CSV: {drc_debug_file}")
    if combined_fasta:
        print(f"   - Combined FASTA: {combined_fasta}")
    else:
        print("   - No successful sequences; combined FASTA was not generated.")

    return {
        "summary_file": summary_file,
        "drc_debug_file": drc_debug_file,
        "combined_fasta": combined_fasta,
        "n_records": len(records),
        "n_success": len(all_final_records),
        "summary_data": summary_data,
        "drc_debug_rows": drc_debug_rows,
    }


In [ ]:
results = run_optimized_batch_pipeline(
    INPUT_FASTA,
    OUTPUT_DIR,
    reference_fasta=REFERENCE_FASTA,
    orientation_k=ORIENTATION_K,
    orientation_chunk_size=ORIENTATION_CHUNK_SIZE,
    orientation_n_chunks=ORIENTATION_N_CHUNKS,
    orientation_step=ORIENTATION_STEP,
    orientation_min_ratio=ORIENTATION_MIN_RATIO,
    drc_coarse_step=DRC_COARSE_STEP,
    drc_fine_step=DRC_FINE_STEP,
    drc_window_size=DRC_WINDOW_SIZE,
    drc_min_identity=DRC_MIN_IDENTITY,
    drc_min_alignment_coverage=DRC_MIN_ALIGNMENT_COVERAGE,
    drc_margin=DRC_MARGIN,
    drc_max_n_fraction=DRC_MAX_N_FRACTION,
    drc_min_opposite_run=DRC_MIN_OPPOSITE_RUN,
    drc_min_reversed_run=DRC_MIN_REVERSED_RUN,
    drc_require_overlap_state=DRC_REQUIRE_OVERLAP_STATE,
    drc_debug=DRC_DEBUG,
    drc_save_debug_report=DRC_SAVE_DEBUG_REPORT,
    itr_search_window=ITR_SEARCH_WINDOW,
    min_itr_length=MIN_ITR_LENGTH,
    min_itr_identity=MIN_ITR_IDENTITY,
    min_itr_score=MIN_ITR_SCORE,
    itr_terminal_tolerance=ITR_TERMINAL_TOLERANCE,
    min_itr_informative_bases=MIN_ITR_INFORMATIVE_BASES,
)
